# 08 - Fact Tables

## Objective

The objective of this notebook is to create and validate the fact tables used for business analysis.

Fact tables contain measurable business events and numerical metrics. They connect the dimension tables to support analytical queries.

## Fact Table Grain

The primary sales fact table follows this grain:

> One row in `fact_sales` represents one product line within an order.

This means that an order can contain multiple rows when multiple products are purchased.

## Source Tables

The fact tables are created using the cleaned Silver tables and the dimension tables.

Main sources include:

- `clean_orders`
- `clean_order_items`
- `clean_order_payments`
- `clean_products`
- `dim_customer`
- `dim_product`
- `dim_order`
- `dim_date`
- `dim_payment`

## Fact Tables Created

### `fact_sales`

Contains product-level sales transactions and measures such as:

- `order_id`
- `customer_id`
- `product_id`
- `price`
- `freight_value`
- Date-related information

### `fact_order_payment`

Contains order-level payment metrics such as:

- `order_id`
- `total_payment_value`
- `payment_transaction_count`
- `max_payment_installments`

## Fact Table Validation

The following validation checks are performed:

- Total row count
- Unique order count
- Unique product count
- NULL foreign keys
- Referential integrity
- Unmatched orders
- Unmatched products
- Unmatched customers
- Duplicate order-item records
- Negative payment values
- Sales and payment totals

## Validation Results

### `fact_sales`

- Total rows: 112,650
- Unique orders: 98,666
- Unique products: 32,951
- NULL orders: 0
- NULL products: 0
- NULL customers: 0
- Unmatched orders: 0
- Unmatched products: 0
- Unmatched customers: 0

### `fact_order_payment`

- Total rows: 99,440
- Unique orders: 99,440
- Negative payment values: 0
- Unmatched payments: 0

## Key Sales Metrics

The `fact_sales` table contains:

- Total product sales: 13,591,643.70
- Total freight: 2,251,909.54
- Total sales value: 15,843,553.24
- Minimum product price: 0.85
- Maximum product price: 6,735

The `fact_order_payment` table contains:

- Total payment value: 16,008,872.12
- Average payment value: 160.99
- Maximum payment value: 13,664.08
- Minimum payment value: 0

## Outcome

The fact tables have been created and validated successfully.

The analytical data model is now ready for the next stage:

**Gold Business Tables → Advanced SQL Analysis → Business Insights**

In [0]:
%sql

DESCRIBE clean_order_items;

####Check the order-item grain

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(order_id, '_', order_item_id)) AS unique_order_items
FROM clean_order_items;

#####1. Validate fact_sales

In [0]:
%sql
SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    COUNT(DISTINCT product_id) AS unique_products
FROM fact_sales;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT_IF(order_id IS NULL) AS null_orders,
    COUNT_IF(product_id IS NULL) AS null_products,
    COUNT_IF(customer_id IS NULL) AS null_customers
FROM fact_sales;

#####2. Check unmatched relationships

In [0]:
%sql
SELECT COUNT(*) AS unmatched_orders
FROM fact_sales f
LEFT JOIN dim_order d
    ON f.order_id = d.order_id
WHERE d.order_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_products
FROM fact_sales f
LEFT JOIN dim_product d
    ON f.product_id = d.product_id
WHERE d.product_id IS NULL;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_customers
FROM fact_sales f
LEFT JOIN dim_customer d
    ON f.customer_id = d.customer_id
WHERE d.customer_id IS NULL;

#####3. Validate sales amounts

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(price) AS total_product_sales,
    SUM(freight_value) AS total_freight,
    SUM(price + freight_value) AS total_sales_value,
    MIN(price) AS min_price,
    MAX(price) AS max_price
FROM fact_sales;

#####4. Validate fact_order_payment

In [0]:
%sql
SELECT
    COUNT(*) AS total_orders,
    SUM(total_payment_value) AS total_payment_value,
    AVG(total_payment_value) AS average_payment_value,
    MAX(total_payment_value) AS max_payment_value,
    MIN(total_payment_value) AS min_payment_value
FROM fact_order_payment;

In [0]:
%sql
SELECT COUNT(*) AS unmatched_payments
FROM fact_order_payment f
LEFT JOIN dim_order d
    ON f.order_id = d.order_id
WHERE d.order_id IS NULL;

In [0]:
%sql
DESCRIBE fact_order_payment;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_orders
FROM fact_order_payment;

In [0]:
%sql
SELECT
    COUNT(*) AS negative_payment_values
FROM fact_order_payment
WHERE total_payment_value < 0;